In [1]:
print("CSV GROQ RAG")

CSV GROQ RAG


In [2]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

True

In [3]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [4]:
llm=ChatGroq(
    model="moonshotai/kimi-k2-instruct",
    temperature=0.4
)

In [5]:
llm.invoke("What is Moon Shot AI").content

'Moonshot AI is a Chinese artificial intelligence company founded in March 2023, focused on developing large language models (LLMs) and advanced AI applications. The company gained prominence with the release of **Kimi**, a highly capable AI assistant known for its **extremely long context window** (supporting up to **2 million Chinese characters** in a single conversation as of 2024).  \n\nKey details about Moonshot AI:  \n- **Founded**: March 2023, Beijing, China.  \n- **Founder**: Yang Zhilin (a former researcher at Google and Carnegie Mellon University).  \n- **Flagship Product**: **Kimi Chat**, a conversational AI similar to ChatGPT but optimized for Chinese users and long-form document analysis.  \n- **Funding**: Valued at **$3 billion** after a $1 billion Series B funding round (Feb 2024), backed by investors like Alibaba and HongShan (formerly Sequoia China).  \n- **Focus Areas**: LLMs, AI agents, and productivity tools for Chinese enterprises.  \n\nMoonshot AI competes with ot

In [7]:
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
embed=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\HP\AppData\Local\Temp\ipykernel_11672\2299495896.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
C:\Users\HP\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
csv_doc_load=CSVLoader("test.csv")

In [11]:
document=csv_doc_load.load()

In [12]:
document

[Document(metadata={'source': 'test.csv', 'row': 0}, page_content='MovieID: 1\nTitle: Inception\nGenre: Sci-Fi\nRating: 8.8\nYear: 2010'),
 Document(metadata={'source': 'test.csv', 'row': 1}, page_content='MovieID: 2\nTitle: The Dark Knight\nGenre: Action\nRating: 9.0\nYear: 2008'),
 Document(metadata={'source': 'test.csv', 'row': 2}, page_content='MovieID: 3\nTitle: Interstellar\nGenre: Sci-Fi\nRating: 8.6\nYear: 2014'),
 Document(metadata={'source': 'test.csv', 'row': 3}, page_content='MovieID: 4\nTitle: Parasite\nGenre: Thriller\nRating: 8.6\nYear: 2019'),
 Document(metadata={'source': 'test.csv', 'row': 4}, page_content='MovieID: 5\nTitle: Avengers: Endgame\nGenre: Action\nRating: 8.4\nYear: 2019'),
 Document(metadata={'source': 'test.csv', 'row': 5}, page_content='MovieID: 6\nTitle: Titanic\nGenre: Romance\nRating: 7.8\nYear: 1997'),
 Document(metadata={'source': 'test.csv', 'row': 6}, page_content='MovieID: 7\nTitle: The Godfather\nGenre: Crime\nRating: 9.2\nYear: 1972'),
 Docume

In [13]:
splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=10)

In [14]:
chunks=splitter.split_documents(document)

In [15]:
for c in chunks:
    print(c)
    print('-'*10)

page_content='MovieID: 1
Title: Inception
Genre: Sci-Fi
Rating: 8.8
Year: 2010' metadata={'source': 'test.csv', 'row': 0}
----------
page_content='MovieID: 2
Title: The Dark Knight
Genre: Action
Rating: 9.0
Year: 2008' metadata={'source': 'test.csv', 'row': 1}
----------
page_content='MovieID: 3
Title: Interstellar
Genre: Sci-Fi
Rating: 8.6
Year: 2014' metadata={'source': 'test.csv', 'row': 2}
----------
page_content='MovieID: 4
Title: Parasite
Genre: Thriller
Rating: 8.6
Year: 2019' metadata={'source': 'test.csv', 'row': 3}
----------
page_content='MovieID: 5
Title: Avengers: Endgame
Genre: Action
Rating: 8.4
Year: 2019' metadata={'source': 'test.csv', 'row': 4}
----------
page_content='MovieID: 6
Title: Titanic
Genre: Romance
Rating: 7.8
Year: 1997' metadata={'source': 'test.csv', 'row': 5}
----------
page_content='MovieID: 7
Title: The Godfather
Genre: Crime
Rating: 9.2
Year: 1972' metadata={'source': 'test.csv', 'row': 6}
----------
page_content='MovieID: 8
Title: The Shawshank Red

In [16]:
vectorstores=Chroma.from_documents(chunks,embed)

In [17]:
retriever=vectorstores.as_retriever(search_type='similarity',search_kwargs={'k':3})

In [18]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory
memory=ConversationBufferMemory()

C:\Users\HP\AppData\Local\Temp\ipykernel_11672\2314992378.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory()


In [19]:
template="""You are a CSV bot. You will get a context and based on it you have to reply make sure reply should not be 
more than 50 words.
Here is question: {question}
Here is context: {context}"""
prompt=PromptTemplate(template=template,input_variables=['question','context'])

In [20]:
csv_bot=RetrievalQA.from_chain_type(llm=llm,memory=memory,retriever=retriever,chain_type_kwargs={'prompt':prompt})

In [21]:
response=csv_bot.invoke('When Avengers Endgame relased and tell me rating too!')

In [22]:
print('--------CSV BOT---------')
print(response['result'])

--------CSV BOT---------
Avengers: Endgame released in 2019 with an 8.4 rating.
